# X-Ray Blue Team Arena

A validated Red Team adversarial image appears in the preview below when you open this notebook from the X-Ray Blue challenge page. Inspect it beside the clean source, experiment locally with randomized smoothing, and submit only a defense configuration. The protected evaluator fetches the authoritative Arena artifact and reruns the defense itself.

**Goal:** keep the clean source useful while correcting or abstaining on the attack. A defense that abstains on everything fails the clean-utility check.

## Quick navigation

[1 · Refresh inbox](#blue-inbox) · [2 · Experiment](#blue-experiment) · [3 · Submit defense](#blue-submit) · [4 · Review history](#blue-review)

The progress strip highlights the current step. The turn banner tells you whether to act or wait.

<!-- ARENA_BLUE_PREVIEW -->
## Red attack preview

No validated Red image has been staged yet. After Red submits an accepted attack, open this notebook from the X-Ray Blue challenge page to see the image here.

## Randomized smoothing

The fixed model classifies noisy views of an input. The majority class and vote fraction summarize the result; `sigma` controls pixel noise, and `num_samples` controls how many views are evaluated. This exercise uses bounded Gaussian noise in pixel space, clipped to [0, 1]. It is a lightweight defense experiment, not a certified robustness guarantee. The evaluator repeats your chosen settings with a deterministic match/attack seed.

In [ ]:
from range_device import select_device, describe_device
device = select_device()
print("Compute device:", describe_device(device))

import red_blue_arena as arena
from IPython.display import display, Markdown, HTML, Image as DisplayImage

identity = arena.identity()
display(identity)
assert identity['role'] == 'blue' or identity.get('mode') == 'solo', 'This workspace is not assigned to Blue.'
display(HTML(arena.progress_header_html('blue', 'inbox')))
display(HTML(arena.turn_indicator_html()))

<a id="blue-inbox"></a>
## 1. Refresh the inbox and download the pending attack

Run this cell again to refresh. The helper verifies the downloaded SHA256 against the Arena record and saves a local copy in this exercise's `inbox/` folder.

In [ ]:
pending = arena.get_pending_attack()
display(HTML(arena.pending_attack_card_html(pending)))
display(HTML(arena.progress_header_html('blue', 'experiment' if pending.get('pending') else 'inbox')))
display(HTML(arena.turn_indicator_html()))
if pending.get('pending'):
    attack_path, attack_info = arena.download_pending_attack()
    source_path = arena.download_source(attack_info['source_id'])
    print('Artifact SHA-256 verified by the Arena helper.')
    display(arena.compare_images(source_path, attack_path))
else:
    print('No Red attack is waiting. Refresh this cell later.')

<a id="blue-experiment"></a>
## 2. Experiment locally

Compare local predictions and smoothing settings before submitting. The protected service repeats the evaluation on the authoritative Red artifact.

In [ ]:
if pending.get('pending'):
    clean_prediction = arena.predict_image(source_path)
    attack_prediction = arena.predict_image(attack_path)
    display(HTML(arena.prediction_card_html(clean_prediction, attack_prediction, role='blue')))
    sigma = 0.05
    num_samples = 100
    abstain_threshold = 0.75
    smoothing = arena.local_randomized_smoothing(
        attack_path, sigma=sigma, num_samples=num_samples, seed=2026
    )
    display(HTML(arena.smoothing_result_card_html(smoothing)))
    display(HTML(arena.progress_header_html('blue', 'submit')))

<a id="blue-submit"></a>
## 3. Submit the defense configuration

The server accepts `sigma` from 0 to 0.1, 10–500 samples, and an abstention threshold from 0 to 1. Arena reruns your settings on its stored Red artifact and records the authoritative outcome. A successful defense returns your X-Ray Blue flag for CTFd. If you lose the output, run `arena.get_blue_flag()` in a new cell.

In [ ]:
if pending.get('pending'):
    result = arena.submit_blue_response(
        attack_info['attack_id'], sigma=sigma,
        num_samples=num_samples, abstain_threshold=abstain_threshold
    )
    display(HTML(arena.blue_result_card_html(result)))
    display(Markdown(arena.round_summary_markdown()))
    display(HTML(arena.progress_header_html('blue', 'review')))
    display(HTML(arena.turn_indicator_html()))
else:
    print('There is no pending attack to answer.')

<a id="blue-review"></a>
## 4. Review the interaction history

Red can now see the defense parameters and adapt for the next sequence. Use the highlighted outcomes and event times to compare rounds.

In [ ]:
display(HTML(arena.turn_indicator_html()))
display(Markdown(arena.round_summary_markdown()))
display(HTML(arena.progress_header_html('blue', 'review')))